# Experiment console

Set a committed runner command and result directory. Run the cells to stream progress, save a log and inspect tables. Downloads require a separate explicit command.

In [ ]:
from pathlib import Path
import subprocess
import sys
import pandas as pd
from IPython.display import display
ROOT = Path.cwd() if (Path.cwd() / 'pyproject.toml').exists() else Path.cwd().parent
COMMAND = [sys.executable, '-u', 'tools/notebook_experiments.py']
RESULTS = ROOT / 'results/sr3-curve-notebook-v1'
print(COMMAND)

In [ ]:
RESULTS.mkdir(parents=True, exist_ok=True)
with (RESULTS / 'notebook-run.log').open('a') as log:
    with subprocess.Popen(COMMAND, cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
        try:
            for line in process.stdout:
                print(line, end=''); log.write(line); log.flush()
            code = process.wait()
            if code:
                raise RuntimeError(f'Runner exited with code {code}; inspect notebook-run.log')
        except KeyboardInterrupt:
            process.terminate(); process.wait()
            raise

In [ ]:
for name in ['summary.json', 'REPORT.md']:
    path = RESULTS / name
    if path.exists(): print(path.read_text())
for name in ['metrics.csv', 'robustness_grid.csv', 'event_regressions.csv']:
    path = RESULTS / name
    if path.exists():
        print(name); display(pd.read_csv(path).head(30))